In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torch_xla
import torch_xla.core.xla_model as xm

from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

In [ ]:
CONFIG = {
    'model_name'  : 'xlm-roberta-base',
    'max_len'     : 128,
    'batch_size'  : 32,
    'epochs'      : 4,
    'lr'          : 2e-5,
    'warmup_ratio': 0.1,
    'test_size'   : 0.20,
    'random_seed' : 42,
    'save_path'   : 'xlmroberta_urdu_best.pt',
}

torch.manual_seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])

# Get TPU device once globally
DEVICE = xm.xla_device()
print(f"TPU Device : {DEVICE}")
print(f"XLA Device type: {xm.xla_device_hw(DEVICE)}")

TPU Device : xla:0
XLA Device type: TPU


In [ ]:
print("\n" + "="*55)
print("STEP 1: Loading and cleaning data...")
print("="*55)

df = pd.read_csv('/content/combined_urdu_news.csv')

label_map = {
    'TRUE': 0, 'TRUE ': 0, 'REAL': 0, 'real': 0, '1': 0,
    'FAKE': 1, 'FAKE ': 1, 'fake': 1, 'fake ': 1,
}
df['Label'] = df['Label'].str.strip()
df['label'] = df['Label'].map(label_map)
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)
df = df.rename(columns={'News Items': 'text'})[['text', 'label']]
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
df = df[df['text'].str.split().str.len() >= 5].reset_index(drop=True)

def urdu_ratio(text):
    urdu = len(re.findall(r'[\u0600-\u06FF]', str(text)))
    return urdu / (len(str(text)) + 1)

df = df[df['text'].apply(urdu_ratio) >= 0.5].reset_index(drop=True)

print(f"Total rows : {len(df)}")
print(f"Real  (0)  : {(df['label']==0).sum()}")
print(f"Fake  (1)  : {(df['label']==1).sum()}")
print(f"Imbalance  : {(df['label']==0).sum()/(df['label']==1).sum():.2f}x")


STEP 1: Loading and cleaning data...
Total rows : 61996
Real  (0)  : 42328
Fake  (1)  : 19668
Imbalance  : 2.15x


In [ ]:
def preprocess_urdu(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u0670]', '', text)
    text = text.replace('\u0649', '\u06CC')
    text = text.replace('\u0643', '\u06A9')
    text = text.replace('\u0647', '\u06BE')
    text = re.sub(r'http\S+|www\S+|@\S+|#\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(preprocess_urdu)
df = df[df['text'].str.strip().str.len() > 0].reset_index(drop=True)

In [ ]:
print("\nSTEP 3: Train/test split...")

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size    = CONFIG['test_size'],
    random_state = CONFIG['random_seed'],
    stratify     = df['label'].values,
)

X_train = df['text'].values[train_idx]
y_train = df['label'].values[train_idx]
X_test  = df['text'].values[test_idx]
y_test  = df['label'].values[test_idx]

print(f"Train : {len(X_train)}  |  Test : {len(X_test)}")


STEP 3: Train/test split...
Train : 49596  |  Test : 12400


In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)

# Move class weights directly to TPU device
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print(f"\nClass weights -> Real: {cw[0]:.4f}  Fake: {cw[1]:.4f}")


Class weights -> Real: 0.7323  Fake: 1.5761


In [ ]:
print("\nSTEP 6: Loading model onto TPU...")

model = XLMRobertaForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels = 2,
)

# Move entire model to TPU device
model = model.to(DEVICE)

# Verify model is on TPU
sample_param = next(model.parameters())
print(f"  Model device : {sample_param.device}")
print(f"  Parameters   : {sum(p.numel() for p in model.parameters()):,}")



STEP 6: Loading model onto TPU...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Model device : xla:0
  Parameters   : 278,045,186


In [ ]:
optimizer    = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
total_steps  = len(train_loader) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)


NameError: name 'train_loader' is not defined

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []

    for step, batch in enumerate(loader):
        # Move each tensor to TPU explicitly
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # THIS IS THE KEY LINE — tells XLA to execute the graph on TPU now
        # Without this, XLA keeps building the graph and never runs it
        xm.mark_step()

        scheduler.step()

        # Move results back to CPU for metric computation
        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())

        if (step + 1) % 100 == 0:
            print(f"    Step {step+1}/{len(loader)} | Loss: {total_loss/(step+1):.4f}", flush=True)

    return (
        total_loss / len(loader),
        f1_score(all_labels, all_preds, average='macro'),
        accuracy_score(all_labels, all_preds),
    )


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)

            # Must call after every forward pass on TPU
            xm.mark_step()

            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.detach().cpu().numpy())

    return (
        total_loss / len(loader),
        f1_score(all_labels, all_preds, average='macro'),
        accuracy_score(all_labels, all_preds),
        all_preds,
        all_labels,
    )

In [ ]:
print("\nSTEP 9: Training...")
print(f"{'Epoch':<8} {'Train Loss':>11} {'Train F1':>10} {'Val Loss':>10} {'Val F1':>8} {'Val Acc':>8}")
print("-" * 58)

best_f1 = 0.0
history = []

for epoch in range(1, CONFIG['epochs'] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['epochs']}")

    train_loss, train_f1, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion
    )
    val_loss, val_f1, val_acc, val_preds, val_labels = eval_epoch(
        model, test_loader, criterion
    )

    history.append({
        'epoch': epoch, 'train_loss': train_loss,
        'train_f1': train_f1, 'val_loss': val_loss,
        'val_f1': val_f1, 'val_acc': val_acc,
    })

    print(f"{epoch:<8} {train_loss:>11.4f} {train_f1:>10.4f} "
          f"{val_loss:>10.4f} {val_f1:>8.4f} {val_acc:>8.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        # xm.save is the TPU-safe version of torch.save
        xm.save(model.state_dict(), CONFIG['save_path'])
        print(f"  -> Saved best model (val F1 = {best_f1:.4f})")
